# 0장 실습 — 환경 준비와 첫 시뮬레이션

교재 0장을 옆에 놓고 이 노트북을 위에서부터 실행합니다.
셀이 전부 오류 없이 돌면 이번 학기 실습 환경이 준비된 것입니다.

마지막 빈칸 하나를 채우면 끝납니다. 20분 정도 걸립니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo

print("저장소:", ROOT)

## 1. 설치 확인 (교재 0.1)

`import smartmob` 이 오류 없이 지나가는지가 설치의 기준입니다.
아래 셀은 그 확인과 함께 이 책이 쓰는 패키지의 버전을 찍습니다.
버전이 `requirements.txt` 와 같으면 교재와 같은 숫자와 그림이 나옵니다.

In [ ]:
import matplotlib
import pandas as pd

import smartmob

banner("설치된 것")
print(f"smartmob     {smartmob.__version__}")
print(f"pandas       {pd.__version__}")
print(f"matplotlib   {matplotlib.__version__}")

그림의 축 이름에 한글이 들어가므로 폰트를 먼저 잡아 둡니다.
이 줄은 그림을 그리는 노트북마다 나옵니다. 출력은 찾아낸 폰트의 이름입니다.

In [ ]:
from smartmob.viz import use_korean_font

use_korean_font()

## 2. 데이터 확인 (교재 0.2)

실습 도시는 하남시입니다. 도로망 파일 둘과 수요 파일 하나가 저장소에 이미 들어 있습니다.
`data_path` 는 저장소의 `data/` 아래 경로를 돌려주는 함수입니다.
노트북을 어디서 열었든 같은 파일을 찾습니다.

In [ ]:
from smartmob.data import data_path

for name in ["road_graph_nodes.parquet", "road_graph_edges.parquet", "demand.csv"]:
    p = data_path(f"hanam/{name}")
    print(f"{name:28s} {p.stat().st_size / 1e6:6.2f} MB")   # st_size 는 바이트 단위 크기입니다

세 파일이 MB 단위로 나오면 데이터가 제자리에 있는 것입니다.
`FileNotFoundError` 가 나면 저장소를 받다가 끊긴 경우가 대부분입니다.

이제 도로망을 읽습니다. `modes=("drive",)` 는 자동차가 다닐 수 있는 도로만 남기라는 뜻입니다.
왜 이 인자가 필요한지는 2장에서 다룹니다. `expect` 는 값이 교재와 같은지 그 자리에서 확인해 줍니다.

In [ ]:
from smartmob.data import load_road_graph

G = load_road_graph("hanam", modes=("drive",))

banner("하남시 자동차 도로망")
expect("노드 수", G.n_nodes, 12_566)
expect("엣지 수", G.n_edges, 28_589)

노드 12,566개, 엣지 28,589개입니다.
교차로와 막다른 길만 노드로 잡고, 그 사이 구간을 엣지 하나로 묶은 결과입니다.
두 값 앞에 `[v]` 가 붙으면 데이터가 제대로 들어온 것입니다.

## 3. 엔진에 연결하기 (교재 0.3)

시뮬레이션을 실제로 돌리는 것은 **DTUMOS** 라는 별도 프로그램입니다. 여기서는 HTTP로 부르기만 합니다.

연결 방법은 셋입니다. 수업 중에는 공용 서버, 엔진 코드를 볼 때는 로컬 Docker, 서버 없이 복습할 때는 녹화본입니다.
`Dtumos()` 는 서버에 붙을 수 있는지 먼저 확인하고, 못 붙으면 `data/fixtures/` 의 녹화본으로 자동 전환합니다.
환경변수 `SMARTMOB_OFFLINE=1` 을 두면 확인 없이 녹화본을 씁니다. 그래서 집에서도 이 노트북이 끝까지 돌아갑니다.

In [ ]:
from smartmob import Dtumos

dt = Dtumos()                       # 서버 주소는 환경변수 SMARTMOB_DTUMOS_URL 에서 읽습니다
print(dt.health())
print("사용 중인 모드:", dt.mode)    # "fixture" 면 녹화본, "live" 면 실서버

`status` 가 `fixture` 로 나오면 녹화본을 쓰는 중입니다.
실서버에 붙었다면 서버가 돌려준 상태가 그대로 나오고 모드는 `live` 입니다. 둘 다 정상입니다.

## 4. 첫 시뮬레이션 (교재 0.4)

하남시에서 저녁 6시부터 자정까지, 택시 80대로 1,000건의 호출을 처리합니다.
`1080` 은 자정부터 센 분이고 18:00입니다. 이 책의 시간은 전부 이 단위입니다.

방금 만든 `dt` 는 `Dtumos` 클래스로 만든 객체입니다.
`dt.run_simulation(...)` 은 이 객체의 메서드이고, 실행 결과는 `sim` 이라는 새 객체에 담깁니다.
`sim.summary()` 처럼 괄호가 붙으면 값을 계산하는 메서드입니다.
`sim.record` 처럼 괄호가 없으면 객체가 이미 가진 자료를 꺼내는 속성입니다.

실행 시간은 모드에 따라 다릅니다. 아래 셀은 먼저 `dt.mode` 를 찍습니다.

- `fixture`: 녹화본을 그대로 읽으므로 바로 끝납니다
- `live`: 실서버가 여섯 시간치 배차를 실제로 계산합니다. 몇 분 걸릴 수 있으니 기다립니다

인자는 녹화본과 똑같이 맞춰 두었습니다. `fixture` 모드에서 값을 바꾸면 `FixtureMissing` 오류가 납니다.
코드가 틀린 것이 아니라 실서버가 필요하다는 뜻입니다. 대수를 바꿔 가며 실험하는 것은 11장과 12장에서 합니다.

In [ ]:
REFERENCE = dict(
    city="hanam",
    mode="taxi",
    fleet_size=80,
    num_passengers=1000,
    time_start=1080,                  # 18:00
    time_end=1440,                    # 24:00
    dispatch_mode="optimization",     # 배차 규칙. 여러 호출과 빈 차를 한꺼번에 짝지어 배정합니다 (10장)
    matrix_mode="street_distance",    # 차량과 승객 사이 거리를 직선이 아니라 도로망 경로로 잽니다 (3장)
    vehicle_capacity=1,               # 차량 한 대에 승객 한 명. 합승이 없습니다
    random_seed=42,                   # 난수 씨앗. 같은 값이면 같은 결과가 나옵니다
)

print("모드:", dt.mode)
sim = dt.run_simulation(**REFERENCE)   # ** 는 딕셔너리를 키워드 인자로 풀어 넣는 문법입니다
sim.summary()

숫자를 하나씩 읽습니다.

- `service_rate` 가 1.0입니다. 990건의 호출이 전부 배차됐습니다.
  1,000건을 넣었는데 990건인 이유가 있습니다.
  마지막 열 건이 23시 56분 이후에 들어온 호출이라, 자정에 끝나는 시뮬레이션이 세지 않습니다.
  이 990이라는 숫자는 뒤의 장에서 계속 나옵니다.
- `avg_waiting_time_min` 이 약 4.1분입니다. 호출하고 차가 올 때까지 평균 4분 걸렸습니다.
- `utilization` 이 약 0.27입니다. 차량이 승객을 태우고 있던 시간이 전체의 27%뿐입니다.

승객은 4분만 기다렸는데 차량의 4분의 3은 놀았습니다.
80대가 너무 많은 것인지, 40대로 줄이면 대기시간이 얼마나 늘어나는지가 1장의 질문입니다.

## 5. 시간에 따라 무슨 일이 있었는가 (교재 0.5)

결과 객체에는 표가 둘 있습니다.

- `sim.record`: 1분마다 한 줄. 컬럼 이름이 `_cnt` 로 끝납니다
- `sim.result`: 같은 시각을 차량 상태별로 더 잘게 나눈 표. 컬럼 이름이 `_num` 으로 끝납니다

이 노트북에서는 `record` 만 보고, `result` 는 1장에서 씁니다. `head()` 는 앞의 다섯 줄만 보여 줍니다.

In [ ]:
sim.record.head()

`time` 이 분 단위 시각이고 첫 줄이 1080, 즉 18:00입니다.
`waiting_passenger_cnt` 는 그 분에 기다리고 있던 승객 수입니다.
`empty_vehicle_cnt` 는 빈 차, `driving_vehicle_cnt` 는 운행 중인 차의 수입니다.
`fail_passenger_cnt` 는 배차를 못 받고 포기한 승객의 누적 수입니다.
시작 직후 빈 차 80대가 몇 분 만에 줄어드는 것이 앞의 다섯 줄에 보입니다.

같은 표를 그림으로 봅니다. `plot_record` 는 `record` 의 컬럼을 시간축 위에 그리는 함수입니다.

In [ ]:
from smartmob.viz import plot_record

plot_record(sim.record);      # 끝의 ; 은 그림 객체의 문자열 출력을 숨깁니다

운행 중 차량이 늘어난 만큼 빈 차가 줄어듭니다. 둘을 더하면 대부분의 시간에 80이 됩니다.

그런데 마지막 30분쯤에서 합이 80보다 작아지고, 대기 승객 수는 오히려 늘어납니다.
근무 시간이 끝난 차량이 하나씩 빠지기 때문입니다. 차량마다 `work_start` 와 `work_end` 가 정해져 있습니다.
두 컬럼을 더해 근무 중 차량 수를 직접 세어 봅니다.

In [ ]:
on_duty = sim.record["empty_vehicle_cnt"] + sim.record["driving_vehicle_cnt"]   # 같은 행끼리 더합니다
print("근무 중 차량 최대:", on_duty.max())
print("근무 중 차량 최소:", on_duty.min())
print("마지막 시각 대기 승객:", sim.record["waiting_passenger_cnt"].iloc[-1])   # iloc[-1] 은 마지막 행

최대는 80대이고 최소는 46대입니다. 자정 직전에는 80대 중 34대가 근무를 마친 상태였습니다.
시뮬레이션이 "차량 80대"를 항상 80대로 다루지 않는다는 뜻입니다. 이것을 알아야 결과를 잘못 읽지 않습니다.

## 6. 빈칸

대기 승객이 가장 많았던 시각과 그때의 인원을 구합니다.
`sim.record` 에서 `waiting_passenger_cnt` 가 가장 큰 행을 찾으면 됩니다.

- `idxmax()` 는 값이 가장 큰 행의 인덱스(행 번호)를 돌려줍니다
- 그 행 번호로 `.loc[행 번호, "time"]` 처럼 같은 행의 다른 컬럼 값을 꺼냅니다
- `minutes_to_hhmm` 이 분을 `HH:MM` 으로 바꿔 줍니다

In [ ]:
from smartmob.data import minutes_to_hhmm

# 여기를 채웁니다. 힌트: idxmax() 로 가장 큰 행의 위치를 얻습니다.
peak_minute = None      # 대기 승객이 가장 많았던 시각 (분)
peak_waiting = None     # 그때의 대기 승객 수 (명)

banner("빈칸 확인")
todo("가장 붐빈 시각", peak_minute, fmt=minutes_to_hhmm)
todo("그때의 대기 승객", peak_waiting)

## 정리

- `load_road_graph("hanam")` 이 도로망을, `Dtumos()` 가 시뮬레이션 엔진을 담당합니다
- 엔진에 못 붙으면 `data/fixtures/` 의 녹화본으로 자동 전환됩니다. `dt.mode` 로 확인합니다
- `sim.summary()` 는 서비스율·평균 대기시간·차량 가동률을, `sim.record` 는 분 단위 시계열을 줍니다
- 1,000건 중 990건만 세는 것은 마지막 열 건이 23:56 이후 호출이기 때문입니다
- 1장 실습에서는 이 결과의 평균 뒤에 무엇이 가려져 있는지 봅니다